In [2]:
import pandas as pd



# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [3]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is CO2 released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

In [4]:
# gCO2/ton-km  - emission factor is CO2 released per ton of commodity per kilometer transported. Different countries use different methods

ton_km = {
    "china": 4,
    "south_korea": 4.5, 
    "vietnam": 7, 
    "india": 6,
    "mexico": 80 
}

## Distance

This code calculates distances between ports for countries beyond the ocean(China, India, South Korea, Vietnam), as well as land distance over the US border for Mexico.

### ISTHS6M

In [5]:
# first, analyze the files from census.gov site. They're encoded using fixed-width ASCII, which makes it hard for conventional Pandas functions to handle it
# for this reason, I had to enlist Claude's help to solve it.

# Correct colspecs for ISTHS6M (State HS6 Imports), record length 258
colspecs = [
    (0, 6),      # commodity (6-digit HS code)
    (6, 10),     # cty_code (4-digit country code)
    (10, 12),    # state (2-letter postal)
    (12, 16),    # year
    (16, 18),    # month
    (18, 33),    # gen_val_mo (general imports total value)
    (33, 48),    # con_val_mo (imports for consumption value)
    (48, 63),    # air_val_mo
    (63, 78),    # air_swt_mo (air shipping weight, kg)
    (78, 93),    # ves_val_mo
    (93, 108),   # ves_swt_mo (vessel shipping weight, kg)
    (108, 123),  # cnt_val_mo (containerized vessel value)
    (123, 138),  # cnt_swt_mo (containerized vessel weight)
    (138, 153),  # gen_val_yr (year-to-date)
    (153, 168),  # con_val_yr
    (168, 183),  # air_val_yr
    (183, 198),  # air_swt_yr
    (198, 213),  # ves_val_yr
    (213, 228),  # ves_swt_yr
    (228, 243),  # cnt_val_yr
    (243, 258),  # cnt_swt_yr
]
names = ['commodity', 'cty_code', 'state', 'year', 'month',
         'gen_val_mo', 'con_val_mo', 'air_val_mo', 'air_swt_mo',
         'ves_val_mo', 'ves_swt_mo', 'cnt_val_mo', 'cnt_swt_mo',
         'gen_val_yr', 'con_val_yr', 'air_val_yr', 'air_swt_yr',
         'ves_val_yr', 'ves_swt_yr', 'cnt_val_yr', 'cnt_swt_yr']

str_cols = ['commodity', 'cty_code', 'state', 'year', 'month']

df_land_imports = pd.read_fwf(
    'data/original/isthsm2501.txt',
    colspecs=colspecs,
    names=names,
    dtype={c: str for c in str_cols},
)

In [6]:
df_land_imports.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,FL,2025,01,57550,57550,0,0,0,...,0,0,57550,57550,0,0,0,0,0,0
1,010121,1220,KY,2025,01,28770,28770,0,0,0,...,0,0,28770,28770,0,0,0,0,0,0
2,010121,1220,MI,2025,01,11500,11500,0,0,0,...,0,0,11500,11500,0,0,0,0,0,0
3,010121,1220,MO,2025,01,10000,10000,0,0,0,...,0,0,10000,10000,0,0,0,0,0,0
4,010121,1220,MT,2025,01,25000,25000,0,0,0,...,0,0,25000,25000,0,0,0,0,0,0


In [7]:
df_land_imports.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449110 entries, 0 to 449109
Data columns (total 21 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   commodity   449110 non-null  object
 1   cty_code    449110 non-null  object
 2   state       449110 non-null  object
 3   year        449110 non-null  object
 4   month       449110 non-null  object
 5   gen_val_mo  449110 non-null  int64 
 6   con_val_mo  449110 non-null  int64 
 7   air_val_mo  449110 non-null  int64 
 8   air_swt_mo  449110 non-null  int64 
 9   ves_val_mo  449110 non-null  int64 
 10  ves_swt_mo  449110 non-null  int64 
 11  cnt_val_mo  449110 non-null  int64 
 12  cnt_swt_mo  449110 non-null  int64 
 13  gen_val_yr  449110 non-null  int64 
 14  con_val_yr  449110 non-null  int64 
 15  air_val_yr  449110 non-null  int64 
 16  air_swt_yr  449110 non-null  int64 
 17  ves_val_yr  449110 non-null  int64 
 18  ves_swt_yr  449110 non-null  int64 
 19  cnt_val_yr  449110 non-

This is the code that could be used to auto-import data. However, the problem with it is that there is poor connection between the API and the website.œ

In [8]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

### PORTHS6MM



In [9]:
# PORTHS6MM (Port HS6 Imports) record layout — record length 230
# Source: census.gov/foreign-trade/reference/products/layouts/dporths6i.html
colspecs = [
    (0, 6),      # commodity (6-digit HS code)
    (6, 10),     # cty_code (Schedule C, 4-digit)
    (10, 12),    # dist_unlade (Schedule D district, 2-digit)
    (12, 14),    # port_unlade (Schedule D port within district, 2-digit)
    (14, 18),    # year
    (18, 20),    # month
    (20, 35),    # gen_val_mo  (general imports value, this month)
    (35, 50),    # air_val_mo
    (50, 65),    # air_swt_mo  (air shipping weight, kg)
    (65, 80),    # ves_val_mo
    (80, 95),    # ves_swt_mo  (vessel shipping weight, kg)
    (95, 110),   # cnt_val_mo  (containerized vessel value)
    (110, 125),  # cnt_swt_mo  (containerized vessel weight)
    (125, 140),  # gen_val_yr  (year-to-date totals begin)
    (140, 155),  # air_val_yr
    (155, 170),  # air_swt_yr
    (170, 185),  # ves_val_yr
    (185, 200),  # ves_swt_yr
    (200, 215),  # cnt_val_yr
    (215, 230),  # cnt_swt_yr
]
names = ['commodity', 'cty_code', 'dist_unlade', 'port_unlade', 'year', 'month',
         'gen_val_mo', 'air_val_mo', 'air_swt_mo', 'ves_val_mo', 'ves_swt_mo',
         'cnt_val_mo', 'cnt_swt_mo',
         'gen_val_yr', 'air_val_yr', 'air_swt_yr', 'ves_val_yr', 'ves_swt_yr',
         'cnt_val_yr', 'cnt_swt_yr']
str_cols = ['commodity', 'cty_code', 'dist_unlade', 'port_unlade', 'year', 'month']

df_ocean_routes = pd.read_fwf(
    'data/original/PORTHS6MM2501.TXT',
    colspecs=colspecs,
    names=names,
    dtype={c: str for c in str_cols},
)

# Build the full 4-digit Schedule D port code (district + port)
df_ocean_routes['port_full'] = df_ocean_routes['dist_unlade'] + df_ocean_routes['port_unlade']

https://www.census.gov/trade/downloads/2025/state_imp/hs6_m/ISTHSM2512.ZIP

In [10]:
import searoute as sr

# coordinates as [lon, lat]
# Shanghai → LA
route = sr.searoute([121.47, 31.23], [-118.27, 33.74])
# in km, following real sea lanes
distance_km = route.properties['length']  

## Weight of imports

Similar to UN Comtrade, but focuses on the USA and is more laconic.

[USA Trade Census](https://usatrade.census.gov/data/Perspective60/Browse/browsetables.aspx?utosid=5687ae9fc5be588295a68da15cd2f1cd&cache=tffv5e)
+ Data Source Selection: State Import Data(Harmonized System)
+ Filters:
    + Measures: Vessel SWT and Air SWT(kg) - The gross weight in kilograms of shipments made by seafaring vessel/airplane at customs
        + No data on land transportation there
    + State: All States
    + Commodity: 845011, 845012, 845019 and 845020(washing machines)
    + Country: India, South Korea, Mexico
    + Time: Jan 2025 - Mar 2026(monthly)

In [11]:
washing_machine_trade_filepath = "data/original/State Imports by HS Commodities_v4.csv"
washing_machine_df = pd.read_csv(washing_machine_trade_filepath, index_col=False, header=2)

In [12]:
washing_machine_df.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg),Unnamed: 5
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174,"1,777,528",NaN
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,NaN,"2,459,827",NaN
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,NaN,"2,268,490",NaN
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,NaN,"2,512,291",NaN
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,NaN,"2,541,012",NaN


# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

## Land-based Distance/Weight Data(Mexico) - US Census

No dataset has everything we need. However, if we gather from enough data sources, we might get a complete picture. But without a good cleaning, inconsistencies will ruin things.



In [13]:
df_land_imports_mex = df_land_imports[df_land_imports["cty_code"] == "2010"]

In [14]:
df_land_imports_mex_v1 = df_land_imports_mex.T.drop_duplicates().T
df_land_imports_mex_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19036 entries, 67 to 448890
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   19036 non-null  object
 1   cty_code    19036 non-null  object
 2   state       19036 non-null  object
 3   year        19036 non-null  object
 4   month       19036 non-null  object
 5   gen_val_mo  19036 non-null  object
 6   con_val_mo  19036 non-null  object
 7   air_val_mo  19036 non-null  object
 8   air_swt_mo  19036 non-null  object
 9   ves_val_mo  19036 non-null  object
 10  ves_swt_mo  19036 non-null  object
 11  cnt_val_mo  19036 non-null  object
 12  cnt_swt_mo  19036 non-null  object
dtypes: object(13)
memory usage: 2.5+ MB


In [15]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1[col] = pd.to_numeric(df_land_imports_mex_v1[col])
df_land_imports_mex_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19036 entries, 67 to 448890
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   19036 non-null  object
 1   cty_code    19036 non-null  object
 2   state       19036 non-null  object
 3   year        19036 non-null  object
 4   month       19036 non-null  object
 5   gen_val_mo  19036 non-null  int64 
 6   con_val_mo  19036 non-null  int64 
 7   air_val_mo  19036 non-null  int64 
 8   air_swt_mo  19036 non-null  int64 
 9   ves_val_mo  19036 non-null  int64 
 10  ves_swt_mo  19036 non-null  int64 
 11  cnt_val_mo  19036 non-null  int64 
 12  cnt_swt_mo  19036 non-null  int64 
dtypes: int64(8), object(5)
memory usage: 2.5+ MB


In [16]:
df_land_imports_mex_wash = df_land_imports_mex_v1[df_land_imports_mex_v1["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash.describe()

,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo
count,2.300000e+01,2.300000e+01,23.0,23.0,23.000000,23.000000,23.000000,23.000000
mean,1.290286e+06,1.289363e+06,0.0,0.0,25014.434783,5846.434783,25014.434783,5846.434783
std,2.927376e+06,2.927754e+06,0.0,0.0,47158.716741,11152.366063,47158.716741,11152.366063
min,1.684900e+04,1.684900e+04,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,4.376000e+04,4.376000e+04,0.0,0.0,0.000000,0.000000,0.000000,0.000000
50%,1.882560e+05,1.882560e+05,0.0,0.0,0.000000,0.000000,0.000000,0.000000
75%,1.176076e+06,1.176076e+06,0.0,0.0,25537.500000,6185.000000,25537.500000,6185.000000
max,1.376476e+07,1.376476e+07,0.0,0.0,146506.000000,33517.000000,146506.000000,33517.000000


So, upon filtering and cleaning data, we recognize that the vast majority of imports of Mexican washing machines into the USA is done through the land, rather than sea or plane. This confirmed my initial hypothesis on Mexico's transformation breakdown and now justifies the plan to calculate solely the inland leg's carbon footprint for imports from Mexico.

Now, we need to understand the distances from each state to state.

In [17]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

In [29]:
df_land_imports_mex_wash.to_csv("data/result/ISTHS6MM_mex.csv")

## Ocean-based Distance/Weight data

In [18]:
asian_countries_codes = ["5700", "5800", "5520", "5330"]
df_asian_routes = df_ocean_routes[df_ocean_routes["cty_code"].isin(asian_countries_codes)]

In [19]:
df_asian_routes_v1 = df_asian_routes.T.drop_duplicates().T


In [20]:
df_asian_routes_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 102949 entries, 108 to 418323
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   commodity    102949 non-null  object
 1   cty_code     102949 non-null  object
 2   dist_unlade  102949 non-null  object
 3   port_unlade  102949 non-null  object
 4   year         102949 non-null  object
 5   month        102949 non-null  object
 6   gen_val_mo   102949 non-null  object
 7   air_val_mo   102949 non-null  object
 8   air_swt_mo   102949 non-null  object
 9   ves_val_mo   102949 non-null  object
 10  ves_swt_mo   102949 non-null  object
 11  cnt_val_mo   102949 non-null  object
 12  cnt_swt_mo   102949 non-null  object
 13  port_full    102949 non-null  object
dtypes: object(14)
memory usage: 15.8+ MB


In [21]:
df_asian_routes_v1.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full
108,010129,5800,39,01,2025,01,10000,10000,700,0,0,0,0,3901
170,010611,5520,10,12,2025,01,7920000,7920000,7360,0,0,0,0,1012
171,010611,5520,54,01,2025,01,7500000,7500000,4284,0,0,0,0,5401
287,010619,5330,17,04,2025,01,3500,3500,57,0,0,0,0,1704
288,010619,5700,04,17,2025,01,370924,370924,369,0,0,0,0,0417


In [22]:
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols_asian:
    df_asian_routes_v1[col] = pd.to_numeric(df_asian_routes_v1[col])
df_asian_routes_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 102949 entries, 108 to 418323
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   commodity    102949 non-null  object
 1   cty_code     102949 non-null  object
 2   dist_unlade  102949 non-null  object
 3   port_unlade  102949 non-null  object
 4   year         102949 non-null  object
 5   month        102949 non-null  object
 6   gen_val_mo   102949 non-null  int64 
 7   air_val_mo   102949 non-null  int64 
 8   air_swt_mo   102949 non-null  int64 
 9   ves_val_mo   102949 non-null  int64 
 10  ves_swt_mo   102949 non-null  int64 
 11  cnt_val_mo   102949 non-null  int64 
 12  cnt_swt_mo   102949 non-null  int64 
 13  port_full    102949 non-null  object
dtypes: int64(7), object(7)
memory usage: 15.8+ MB


In [23]:
df_asian_routes_wash = df_asian_routes_v1[df_asian_routes_v1["commodity"].isin(["845011", "845020"])]
df_asian_routes_wash.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full
273372,845011,5520,10,03,2025,01,1476672,0,0,1476672,215096,1476672,215096,1003
273373,845011,5520,13,03,2025,01,3888,0,0,3888,740,3888,740,1303
273374,845011,5520,14,01,2025,01,8976,0,0,8976,2224,8976,2224,1401
273375,845011,5520,17,03,2025,01,153763,0,0,153763,21411,153763,21411,1703
273376,845011,5520,27,04,2025,01,19801,0,0,19801,4583,19801,4583,2704


In [ ]:
asian_origins = {
    "5700": ("Shanghai", [31.230, 121.470]),
    "5800": ("Busan",    [35.100, 129.040]),
    "5520": ("Cat Lai",  [10.780, 106.780]),
    "5330": ("Mundra",   [22.740, 69.710]),
}

# US ports of unlading (Schedule D 4-digit code -> [lon, lat])
us_ports = {
    "1003": [ -74.165,  40.685],  # Newark, NJ (Port Newark/Elizabeth)
    "1303": [ -76.535,  39.265],  # Baltimore, MD (Seagirt Marine Terminal)
    "1401": [ -76.330,  36.880],  # Norfolk/Newport News, VA
    "1601": [ -79.911,  32.852],  # Charleston, SC
    "1703": [ -81.140,  32.115],  # Savannah, GA (Garden City Terminal)
    "1801": [ -82.456,  27.910],  # Tampa, FL
    "1803": [ -81.534,  30.395],  # Jacksonville, FL (Blount Island/Dames Point)
    "1901": [ -88.040,  30.690],  # Mobile, AL
    "2002": [ -90.060,  29.940],  # New Orleans, LA
    "2704": [-118.265,  33.733],  # Los Angeles, CA
    "2709": [-118.216,  33.755],  # Long Beach, CA
    "2809": [-122.395,  37.795],  # San Francisco, CA
    "2811": [-122.290,  37.795],  # Oakland, CA
    "2904": [-122.755,  45.585],  # Portland, OR
    "3001": [-122.330,  47.610],  # Seattle, WA
    "3002": [-122.430,  47.275],  # Tacoma, WA
    "3126": [-149.890,  61.235],  # Anchorage, AK
    "3201": [-157.870,  21.310],  # Honolulu, HI
    "3901": [ -87.620,  41.880],  # Chicago, IL (Great Lakes / inland)
    "4101": [ -81.700,  41.500],  # Cleveland, OH (Great Lakes)
    "4909": [ -66.100,  18.450],  # San Juan, PR
    "5201": [ -80.165,  25.778],  # Miami, FL (PortMiami)
    "5203": [ -80.116,  26.090],  # Port Everglades, FL
    "5206": [ -80.290,  25.796],  # Miami International AIRPORT — exclude from sea analysis
    "5301": [ -95.020,  29.733],  # Houston, TX
}


In [ ]:
asian_origins["5700"][1]

[31.23, 121.47]

In [ ]:
import searoute as sr

# coordinates as [lon, lat]
# Shanghai → LA
route = sr.searoute([121.47, 31.23], [-118.27, 33.74])
# in km, following real sea lanes
distance_km = route.properties['length']  

In [ ]:
df_asian_routes_wash["distance"] = sr.searoute(asian_origins[df_asian_routes_wash["cty_code"]][1], df_asian_routes_wash["cty_code"]][1])

In [30]:
df_asian_routes_wash.to_csv("data/result/PORTHS6MM_asian.csv")

## Weight Data(USA Trade Online)

Already well-organized, the weight data does not need much cleaning.

In [24]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [25]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [26]:
washing_machine_df_v1[washing_machine_df_v1["Country"]=="Mexico"].head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
32,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,January 2025,0.0,55246.0
33,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,February 2025,0.0,74445.0
34,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,March 2025,0.0,239852.0
35,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,April 2025,0.0,175512.0
36,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,May 2025,0.0,199598.0


In [27]:
washing_machine_df_v1["Time"].value_counts()

April 2025            9
June 2025             9
July 2025             9
October 2025          9
December 2025         9
2026 through March    9
January 2026          9
January 2025          8
February 2025         8
March 2025            8
May 2025              8
August 2025           8
September 2025        8
November 2025         8
February 2026         8
March 2026            8
Name: Time, dtype: int64

In [28]:
washing_machine_df_v1["Air SWT (kg)"].mean()

171.28148148148148